In [1]:
import pandas as pd 
import numpy as np
import statsmodels.api as sm
import glob
import os
import matplotlib.pyplot as plt
from pandas.plotting import table
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle
from reportlab.lib.pagesizes import landscape, letter
from reportlab.lib import colors
from scipy.stats import t

Using Daily data resample weekly data

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import t

###############################################################################
# 辅助函数：根据 p 值格式化系数（附加显著性星号），保留4位小数
###############################################################################
def format_coef(coef, p):
    if np.isnan(coef) or np.isnan(p):
        return f"{coef:.4f}"
    star = ""
    if p < 0.01:
        star = "***"
    elif p < 0.05:
        star = "**"
    elif p < 0.1:
        star = "*"
    return f"{coef:.4f}{star}"

###############################################################################
# 0. 日频数据聚合为周频数据（使用 ISO 8601 周规则）
###############################################################################
def aggregate_daily_to_weekly(df):
    df = df.rename(columns={"time_open": "Date"})
    df["Date"] = pd.to_datetime(df["Date"])
    iso = df["Date"].dt.isocalendar()
    df["iso_year"] = iso.year
    df["iso_week"] = iso.week

    df = df.dropna()
    df = df[(df["volume"] > 0) & (df["close"] > 0)]
    df = df.dropna(subset=["market_cap"])
    df = df[df["market_cap"] > 1_000_000]

    df["price_vol"] = df["close"] * df["volume"]
    df["log_price_vol"] = np.log(df["price_vol"])
    
    agg_dict = {
        "Date": "max",
        "close": "last",
        "high": "max",
        "market_cap": "last",
        "volume": "mean",
        "price_vol": "mean",
        "log_price_vol": "std"
    }
    weekly = df.groupby(["id", "iso_year", "iso_week"], as_index=False).agg(agg_dict)
    
    weekly.rename(columns={
        "Date": "Date_max",
        "close": "close_last",
        "high": "high_max",
        "market_cap": "market_cap_last",
        "volume": "volume_mean",
        "price_vol": "price_vol_mean",
        "log_price_vol": "log_price_vol_std"
    }, inplace=True)
    weekly["MCAP"] = np.log(weekly["market_cap_last"])
    weekly["PRC"] = np.log(weekly["close_last"])
    weekly["MAXDPRC"] = np.log(weekly["high_max"])
    weekly["PRCVOL"] = np.log(weekly["price_vol_mean"])
    weekly.rename(columns={"log_price_vol_std": "STDPRCVOL"}, inplace=True)
    
    weekly.sort_values(["id", "Date_max"], inplace=True)
    weekly["rtn"] = weekly.groupby("id")["close_last"].transform(lambda x: np.log(x).diff())
    weekly.dropna(subset=["rtn"], inplace=True)
    weekly["Date"] = weekly.apply(lambda row: pd.Timestamp.fromisocalendar(int(row["iso_year"]), int(row["iso_week"]), 7), axis=1)
    
    return weekly

###############################################################################
# 1. 读取并聚合加密货币日频数据，并合并 classification
###############################################################################
def load_crypto_data(folder_path):
    file_paths = sorted(glob.glob(os.path.join(folder_path, "crypto_full_*_daily_data.csv")))
    daily_df = pd.concat((pd.read_csv(fp) for fp in file_paths), ignore_index=True)
    weekly_df = aggregate_daily_to_weekly(daily_df)

    # 读取分类数据，并映射：brown->1, green->0
    df_class = pd.read_csv(r"D:\桌面\绿色货币\Code\Add Green vs Brown Variable\crypto_classification.csv")
    df_class['classification'] = df_class['classification'].map({'brown': 1, 'green': 0})
    
    # 合并 classification 列（按 id）
    weekly_df = pd.merge(weekly_df, df_class[['id', 'classification']], on='id', how='inner')
    weekly_df['Date'] = pd.to_datetime(weekly_df['Date'])
    
    # 特殊调整：例如 id==1027 且日期在2022-09-15之前，赋值为1
    weekly_df.loc[(weekly_df['id'] == 1027) & (weekly_df['Date'] < '2022-09-15'), 'classification'] = 1
    return weekly_df

def preprocess_crypto_data(df):
    df = df.sort_values(["id", "Date"]).reset_index(drop=True)
    df = df.dropna(subset=["rtn"])
    if "MCAP" not in df.columns:
        df["MCAP"] = np.log(df["market_cap_last"])
    if "PRC" not in df.columns:
        df["PRC"] = np.log(df["close_last"])
    if "MAXDPRC" not in df.columns:
        df["MAXDPRC"] = np.log(df["high_max"]) if "high_max" in df.columns else np.log(df["close_last"])
    
    df["r_1_0"] = df.groupby("id")["rtn"].shift(1)
    df["r_2_0"] = df.groupby("id")["rtn"].shift(2)
    df["r_3_0"] = df.groupby("id")["rtn"].shift(3)
    df["r_4_0"] = df.groupby("id")["rtn"].shift(4)
    df["r_4_1"] = df[["r_1_0", "r_2_0", "r_3_0", "r_4_0"]].mean(axis=1)
    
    df["volume"] = df["volume_mean"]
    return df

###############################################################################
# 2. 读取国债日频数据，转换为周频数据，并计算超额收益
###############################################################################
def merge_riskfree_and_compute_excess(df, rf_path):
    rf_data = pd.read_csv(rf_path)
    rf_data["Date"] = pd.to_datetime(rf_data["Date"])
    rf_data["Rf"] = rf_data["1 Mo"] / 100
    rf_data = rf_data.set_index("Date").resample("W").mean().reset_index()
    df = pd.merge_asof(df.sort_values("Date"),
                       rf_data[["Date", "Rf"]].sort_values("Date"),
                       on="Date", direction="backward")
    df["excess_return"] = df["rtn"] - df["Rf"]
    return df

###############################################################################
# 3. 构造三大因子：CMKT, CSMB, CMOM（市值加权）
###############################################################################
def weighted_mean(g, value_col="excess_return", weight_col="market_cap_last"):
    return (g[value_col] * g[weight_col]).sum() / g[weight_col].sum()

def build_3factors(df):
    CMKT = df.groupby("Date").apply(
        lambda g: np.sum(g["excess_return"] * g["market_cap_last"]) / np.sum(g["market_cap_last"])
    ).rename("CMKT")
    
    def assign_size(g):
        q30 = g["market_cap_last"].quantile(0.3)
        q70 = g["market_cap_last"].quantile(0.7)
        g = g.copy()
        g["Size_Group"] = "Big"
        g.loc[g["market_cap_last"] <= q30, "Size_Group"] = "Small"
        g.loc[(g["market_cap_last"] > q30) & (g["market_cap_last"] <= q70), "Size_Group"] = "Middle"
        return g
    df_size = df.groupby("Date").apply(assign_size).reset_index(drop=True)
    small_ret = df_size[df_size["Size_Group"] == "Small"].groupby("Date").apply(
        lambda g: weighted_mean(g, "excess_return", "market_cap_last")
    )
    big_ret = df_size[df_size["Size_Group"] == "Big"].groupby("Date").apply(
        lambda g: weighted_mean(g, "excess_return", "market_cap_last")
    )
    CSMB = (small_ret - big_ret).rename("CSMB")
    
    def assign_size_mom(g):
        q30 = g["market_cap_last"].quantile(0.3)
        q70 = g["market_cap_last"].quantile(0.7)
        g = g[(g["market_cap_last"] <= q30) | (g["market_cap_last"] > q70)].copy()
        g["Size_Group"] = np.where(g["market_cap_last"] <= q30, "Small", "Big")
        return g
    df_mom = df.groupby("Date").apply(assign_size_mom).reset_index(drop=True)
    df_mom["momentum"] = df_mom.groupby("id")["rtn"].transform(
        lambda x: (1 + x).rolling(window=3, min_periods=3).apply(np.prod, raw=True) - 1
    )
    df_mom = df_mom.dropna(subset=["momentum"])
    def assign_mom(sub):
        sub = sub.sort_values("momentum").reset_index(drop=True)
        n = len(sub)
        if n < 3:
            sub["MomGroup"] = np.nan
            return sub
        c1 = int(np.floor(n * 0.3))
        c2 = int(np.floor(n * 0.7))
        sub.loc[:c1-1, "MomGroup"] = "Low"
        sub.loc[c1:c2-1, "MomGroup"] = "Mid"
        sub.loc[c2:, "MomGroup"] = "High"
        return sub
    df_mom = df_mom.groupby(["Date", "Size_Group"], group_keys=False).apply(assign_mom)
    def compute_cmom(g):
        sH = weighted_mean(g[(g["Size_Group"] == "Small") & (g["MomGroup"] == "High")],
                             "rtn", "market_cap_last")
        bH = weighted_mean(g[(g["Size_Group"] == "Big") & (g["MomGroup"] == "High")],
                             "rtn", "market_cap_last")
        sL = weighted_mean(g[(g["Size_Group"] == "Small") & (g["MomGroup"] == "Low")],
                             "rtn", "market_cap_last")
        bL = weighted_mean(g[(g["Size_Group"] == "Big") & (g["MomGroup"] == "Low")],
                             "rtn", "market_cap_last")
        if pd.isnull(sH) or pd.isnull(bH) or pd.isnull(sL) or pd.isnull(bL):
            return np.nan
        return 0.5 * (sH + bH) - 0.5 * (sL + bL)
    CMOM = df_mom.groupby("Date").apply(compute_cmom).rename("CMOM")
    factors = pd.concat([CMKT, CSMB, CMOM], axis=1)
    return factors

###############################################################################
# 4. 构造10个多空策略（Q5-Q1），市值加权计算多空组合收益差
###############################################################################
def build_long_short_strategies(df):
    feats_10 = ["MCAP", "PRC", "MAXDPRC", "r_1_0", "r_2_0", "r_3_0", "r_4_0", "r_4_1", "PRCVOL", "STDPRCVOL"]
    ls_list = []
    for fac in feats_10:
        ls_diff = []
        for date_i, sub in df.groupby("Date"):
            sub_valid = sub.dropna(subset=[fac, "excess_return", "market_cap_last"])
            if len(sub_valid) < 5:
                continue
            sub_valid = sub_valid.sort_values(fac)
            q = len(sub_valid) // 5
            if q < 1:
                continue
            low_ret = weighted_mean(sub_valid.iloc[:q], "excess_return", "market_cap_last")
            high_ret = weighted_mean(sub_valid.iloc[-q:], "excess_return", "market_cap_last")
            ls_diff.append((date_i, high_ret - low_ret))
        tmp = pd.DataFrame(ls_diff, columns=["Date", fac + "_LS"]).set_index("Date")
        ls_list.append(tmp)
    ls_merged = pd.concat(ls_list, axis=1).dropna()
    return ls_merged

###############################################################################
# 5. 回归函数：利用三因子回归（返回数值版大表，便于比较 alpha）
###############################################################################
def run_ols(data, y_col, x_cols):
    X = sm.add_constant(data[x_cols])
    y = data[y_col]
    model = sm.OLS(y, X, missing='drop').fit()
    return model

def run_regression_table_numeric(ls_merged, factors):
    common_idx = ls_merged.index.intersection(factors.index)
    ls_merged = ls_merged.loc[common_idx]
    factors   = factors.loc[common_idx]
    data_all = ls_merged.join(factors, how="inner")
    
    feats_10 = ["MCAP", "PRC", "MAXDPRC", "r_1_0", "r_2_0", "r_3_0", "r_4_0", "r_4_1", "PRCVOL", "STDPRCVOL"]
    results_rows = []
    for fac in feats_10:
        y_var = fac + "_LS"
        # 采用三种规格（Spec 1: CMKT+CSMB, Spec 2: CMKT+CMOM, Spec 3: CMKT+CSMB+CMOM）
        for spec, x_cols in enumerate([["CMKT", "CSMB"], ["CMKT", "CMOM"], ["CMKT", "CSMB", "CMOM"]], start=1):
            model = run_ols(data_all, y_var, x_cols)
            alpha = model.params.get("const", np.nan)
            se_alpha = model.bse.get("const", np.nan)
            p_alpha = model.pvalues.get("const", np.nan)
            row = {
                "Strategy": fac,
                "(Spec)": spec,
                "alpha": alpha,
                "se_alpha": se_alpha,
                "p_alpha": p_alpha,
                "R^2": model.rsquared,
                "N": model.nobs
            }
            results_rows.append(row)
    results_df = pd.DataFrame(results_rows).sort_values(["Strategy", "(Spec)"]).reset_index(drop=True)
    return results_df

###############################################################################
# 6. 主函数：分组计算大表，并比较两组 alpha 差异（保存时保留4位小数，并加星号）
###############################################################################
def main():
    # 读取日频数据并聚合
    folder_path = r"D:\桌面\绿色货币\数据总库\R数据抓取加密货币数据"
    df = load_crypto_data(folder_path)
    
    # 数据预处理
    df = preprocess_crypto_data(df)
    
    # 合并国债数据并计算超额收益
    rf_path = r"D:\桌面\绿色货币\数据总库\宏观经济数据\美国国债\us_treasury_daliy_1M.csv"
    df = merge_riskfree_and_compute_excess(df, rf_path)
    
    # 根据 classification 分为两组（1 与 0）
    df_class1 = df[df["classification"] == 1]
    df_class0 = df[df["classification"] == 0]
    
    # 对每组分别计算多空策略大表和三因子
    ls_merged_1 = build_long_short_strategies(df_class1)
    factors_1 = build_3factors(df_class1)
    table1 = run_regression_table_numeric(ls_merged_1, factors_1)
    
    ls_merged_0 = build_long_short_strategies(df_class0)
    factors_0 = build_3factors(df_class0)
    table0 = run_regression_table_numeric(ls_merged_0, factors_0)
    
    # 添加组别标识
    table1["Group"] = 1
    table0["Group"] = 0
    
    # 将两组大表按 Strategy 和 (Spec) 合并
    combined = pd.merge(table1, table0, on=["Strategy", "(Spec)"], suffixes=("_1", "_0"))
    combined["alpha_diff"] = combined["alpha_1"] - combined["alpha_0"]
    combined["se_diff"] = np.sqrt(combined["se_alpha_1"]**2 + combined["se_alpha_0"]**2)
    combined["t_stat"] = combined["alpha_diff"] / combined["se_diff"]
    # 自由度取两组中较小的样本量减去参数个数（此处简单近似）
    combined["df"] = combined[["N_1", "N_0"]].min(axis=1) - 3  
    combined["p_value"] = 2 * (1 - t.cdf(np.abs(combined["t_stat"]), df=combined["df"]))
    
    # 使用 format_coef 函数对 alpha 结果进行格式化（带显著性星号）
    combined["alpha_1_fmt"] = combined.apply(lambda row: format_coef(row["alpha_1"], row["p_alpha_1"]), axis=1)
    combined["alpha_0_fmt"] = combined.apply(lambda row: format_coef(row["alpha_0"], row["p_alpha_0"]), axis=1)
    combined["alpha_diff_fmt"] = combined.apply(lambda row: format_coef(row["alpha_diff"], row["p_value"]), axis=1)
    
    # 对其他数值型列（如 t_stat, p_value）保留4位小数
    for col in ["p_alpha_1", "p_alpha_0", "t_stat", "p_value"]:
        combined[col] = combined[col].map(lambda x: f"{x:.4f}")
    
    # 输出各策略下两组 alpha 及差异检验结果（显示格式化结果）
    cols_to_show = [
        "Strategy", "(Spec)",
        "alpha_1_fmt", "p_alpha_1",
        "alpha_0_fmt", "p_alpha_0",
        "alpha_diff_fmt", "t_stat", "p_value",
        "df", "N_1", "N_0"
    ]
    print("===== 按分类分组的多因子回归大表比较（含星号） =====")
    print(combined[cols_to_show])
    Green_Alpha = combined[cols_to_show]
    Green_Alpha.to_csv('2014-2024_Green_Alpha.csv', index=False)
    # 保存结果到 CSV，所有数字均以字符串格式保存，保留4位小数和星号信息
    combined.to_csv('2014-2024_Green_vs_Brown_withStars.csv', index=False)
    
    print("\n注：若 p_value < 0.05，则认为两组 alpha 差异在统计上显著。")

if __name__ == "__main__":
    main()


C:\Users\why97\AppData\Local\Temp\ipykernel_6076\146051133.py:134: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMKT = df.groupby("Date").apply(
C:\Users\why97\AppData\Local\Temp\ipykernel_6076\146051133.py:146: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_size = df.groupby("Date").apply(assign_size).reset_index(drop=True)
C:\Users\why97\AppData\Local\Temp\ipykernel_6076\146051133.py:147: DeprecationWa

===== 按分类分组的多因子回归大表比较（含星号） =====
     Strategy  (Spec) alpha_1_fmt p_alpha_1 alpha_0_fmt p_alpha_0  \
0     MAXDPRC       1  -0.0178***    0.0045  -0.0414***    0.0009   
1     MAXDPRC       2    0.0207**    0.0281  -0.0923***    0.0000   
2     MAXDPRC       3      0.0112    0.2158  -0.0795***    0.0000   
3        MCAP       1   0.0290***    0.0000   0.0206***    0.0000   
4        MCAP       2   0.0303***    0.0000      0.0007    0.9354   
5        MCAP       3   0.0193***    0.0000   0.0100***    0.0057   
6         PRC       1  -0.0183***    0.0036     -0.0004    0.9738   
7         PRC       2    0.0208**    0.0273  -0.0539***    0.0012   
8         PRC       3      0.0111    0.2150  -0.0509***    0.0020   
9      PRCVOL       1  -0.0443***    0.0000     -0.0026    0.8457   
10     PRCVOL       2      0.0038    0.8104   0.0622***    0.0040   
11     PRCVOL       3     -0.0105    0.4937   0.0662***    0.0020   
12  STDPRCVOL       1   0.1654***    0.0000   0.1672***    0.0000   
1

C:\Users\why97\AppData\Local\Temp\ipykernel_6076\146051133.py:191: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  CMOM = df_mom.groupby("Date").apply(compute_cmom).rename("CMOM")



**Classification**: 

POW = Brown

None-POW = Green

Group1: Brown

Group0: Green

| **Column Name**    | **Description**                                                                                                                                     |
|--------------------|-----------------------------------------------------------------------------------------------------------------------------------------------------|
| **Strategy**       | The feature on which the long-short strategy is based (e.g., MCAP, PRC, r_1_0, etc.).                                                              |
| **(Spec)**         | The regression specification number. Different specs indicate different factor combinations (e.g.,(1)CMKT+CSMB, (2)CMKT+CMOM, (3)CMKT+CSMB+CMOM).       |
| **alpha_1_fmt**    | The estimated intercept (alpha) for Group 1 (classification = 1), formatted to 4 decimal places with significance stars (e.g., ***).             |
| **p_alpha_1**      | The p-value for the intercept (alpha) of Group 1.                                                                                                   |
| **alpha_0_fmt**    | The estimated intercept (alpha) for Group 0 (classification = 0), formatted to 4 decimal places with significance stars.                           |
| **p_alpha_0**      | The p-value for the intercept (alpha) of Group 0.                                                                                                   |
| **alpha_diff_fmt** | The difference between Group 1 and Group 0 alphas (alpha_1 - alpha_0), formatted to 4 decimal places with significance stars based on its p-value. |
| **t_stat**         | The t-statistic for testing the difference between the two intercepts (alpha values).                                                               |
| **p_value**        | The p-value associated with the intercept difference; a p_value < 0.05 indicates a statistically significant difference.                           |
| **df**             | Degrees of freedom used in the test, typically computed as the smaller sample size minus the number of estimated parameters.                         |
| **N_1**            | The effective sample size (number of observations) for Group 1 (classification = 1).                                                               |
| **N_0**            | The effective sample size (number of observations) for Group 0 (classification = 0).                                                               |